In [30]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 1
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
# -----------------------------------------------------------------------------
def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0):
    n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION
#    Each chirp step is built from NUM_TONES=4 tone slices, kept as SEPARATE
#    envelope buffers. Each tone's buffer is looped (mode='periodic')
#    continuously for its own share of the step-hold time -- tone 0 plays
#    uninterrupted for ratio_0 * STEP_HOLD_US, then tone 1, etc. Only once
#    all 4 tones have had their turn does the chirp step advance.
#
#    Buffer CONTENT length is now ratio-weighted per tone (each tone gets
#    as large a buffer as its share of the memory budget allows), NOT
#    forced uniform across tones. This works because the hardware hop
#    between tone buffers reloads BOTH the address register (mathi) AND
#    the packed length/mode register (regwi) before each tone's pulse --
#    both are pure compile-time constants (see get_mode_code / set_reg in
#    qick_lib, which is a straight wrapper around regwi), so per-tone
#    buffer lengths cost nothing extra in tProc program memory. The only
#    thing that must stay constant is buffer length ACROSS STEPS for a
#    given tone index j (true here, since tone_durations_s doesn't depend
#    on step i) -- lengths may still differ freely BETWEEN tones.
# -----------------------------------------------------------------------------
CHIRP_SPAN_HZ = 350e6
CHIRP_OFFSET_STOP_HZ = 0.0                                       # ends AT the base tones
CHIRP_OFFSET_START_HZ = CHIRP_OFFSET_STOP_HZ - CHIRP_SPAN_HZ      # starts 350 MHz below them

NUM_STEPS = 44
AMPLITUDE = 0.5 #0.11

BASE_TONES_HZ = np.array([76.25e6, 0, -122.92e6+76.25e6, -147.82e6+76.25e6])
TONE_RATIOS = np.array([0.337, 0.167, 0.288, 0.208])

# BASE_TONES_HZ = np.array([0])
# TONE_RATIOS = np.array([1.0])

BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)

TOTAL_SWEEP_S = 6e-3
STEP_HOLD_S = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
CYCLE_S = max_feasible_total_s / (NUM_STEPS + 1)   # per-step time budget (all tones combined)

tone_fractions = TONE_RATIOS / TONE_RATIOS.sum()

# Ratio-weighted tone durations -- used for BOTH the sweep buffers and the
# trap buffer now. Each tone gets its own share of the per-step budget,
# maximizing buffer length (and thus minimizing periodic-loop-wrap
# discontinuities) within the available envelope memory.
tone_durations_s = tone_fractions * CYCLE_S

# Real per-tone hold time on hardware each step (used only for phase
# bookkeeping between tones/steps -- NOT for buffer content generation,
# since the buffer is looped many times to fill this much longer hold).
hold_durations_uniform_s = STEP_HOLD_S * tone_fractions

print(f"Composite per-step time budget: {CYCLE_S*1e9:.2f} ns")
for i, (bt, r, td) in enumerate(zip(BASE_TONES_HZ, TONE_RATIOS, tone_durations_s)):
    hold_us = STEP_HOLD_US * (r / TONE_RATIOS.sum())
    print(f"  tone {i} ({bt/1e6:+.1f} MHz offset): ratio {r} -> buffer {td*1e9:.2f} ns, "
          f"held {hold_us:.3f} us per step")

def build_multitone_pieces(chirp_offset_hz, phase0, gen_durations_s, hold_durations_s):
    """Build the per-tone envelope pieces for one chirp step/stage, kept
    SEPARATE (not concatenated).

    gen_durations_s : per-tone buffer CONTENT duration (how much is
        actually synthesized into the envelope table).
    hold_durations_s : per-tone REAL hardware hold duration (how long the
        buffer is actually looped for on hardware) -- used only to advance
        the running phase correctly between tones/steps, since the two can
        differ (buffer is generated short, then periodic-looped much
        longer to fill its hold time).

    Returns (pieces, phase, piece_lens): pieces is a list of (idata,qdata)
    in playback order; phase is the running phase after all NUM_TONES
    tones (phase-continuous into the next step/stage); piece_lens are the
    per-piece sample counts.
    """
    pieces = []
    lens = []
    phase = phase0
    for base_tone, gen_dur, hold_dur in zip(BASE_TONES_HZ, gen_durations_s, hold_durations_s):
        f = chirp_offset_hz + base_tone
        y, _, n_samples = serrodyne_tone(f, gen_dur, ENV_SR, amplitude=AMPLITUDE, phase0=phase)
        pad = (-len(y)) % samps_per_clk
        if pad:
            y = np.concatenate([y, np.zeros(pad)])
        idata = np.round(y * maxv).astype(np.int16)
        qdata = np.zeros(len(y), dtype=np.int16)
        pieces.append((idata, qdata))
        lens.append(len(idata))
        # advance phase by the REAL elapsed hold time on hardware, not by
        # the (generally much shorter) buffer-generation duration
        phase = (phase + 2 * np.pi * f * hold_dur) % (2 * np.pi)
    return pieces, phase, lens

def build_multitone_buffer(chirp_offset_hz, phase0, durations_s):
    """Same as build_multitone_pieces, but concatenated into one composite
    buffer. Used for the trap stage. The trap buffer is played for exactly
    `durations_s` per tone (no address-stepping / periodic-loop-hold
    mismatch), so gen duration == hold duration here."""
    pieces, phase, lens = build_multitone_pieces(chirp_offset_hz, phase0, durations_s, durations_s)
    idata = np.concatenate([p[0] for p in pieces])
    qdata = np.concatenate([p[1] for p in pieces])
    return idata, qdata, phase, lens

chirp_offsets_hz = np.linspace(CHIRP_OFFSET_START_HZ, CHIRP_OFFSET_STOP_HZ, NUM_STEPS)
maxv = soccfg.get_maxv(GEN_CH)

# idata_pieces[i][j] / qdata_pieces[i][j] = tone j's envelope for chirp step i.
# Length is constant across steps i for a fixed tone j (tone_durations_s
# doesn't depend on i), but differs freely between tones.
idata_pieces = []
qdata_pieces = []
phase = 0.0
piece_lens = None
for offset in chirp_offsets_hz:
    pieces, phase, piece_lens = build_multitone_pieces(
        offset, phase, tone_durations_s, hold_durations_uniform_s)
    idata_pieces.append([p[0] for p in pieces])
    qdata_pieces.append([p[1] for p in pieces])

tone_buf_lens = [len(idata_pieces[0][j]) for j in range(NUM_TONES)]
samples_per_step = sum(tone_buf_lens)
total_samples = samples_per_step * NUM_STEPS

print(f"Per-tone sweep buffer lengths: {tone_buf_lens} samples "
      f"({[round(n/samps_per_clk,1) for n in tone_buf_lens]} fabric cycles)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN
for j, n in enumerate(tone_buf_lens):
    assert n % samps_per_clk == 0, f"tone {j} buffer length {n} not a multiple of samps_per_clk"
    assert n >= 3 * samps_per_clk, (
        f"tone {j} buffer is only {n} samples ({n/samps_per_clk:.1f} fabric cycles) -- "
        f"hardware needs >= 3. Reduce NUM_STEPS, or make TONE_RATIOS less extreme."
    )

# NOTE: this still registers NUM_STEPS*NUM_TONES distinct named envelope
# buffers on the generator. The PROGRAM (compiled tProc instructions) is a
# real hardware loop over the NUM_STEPS steps, not unrolled in Python, so
# it stays well under the tProc program-memory limit regardless of how
# many distinct envelope buffers are registered. If your board's QICK
# build enforces a hard cap on the number of distinct envelope table
# entries per generator, check that it comfortably exceeds this count.
print(f"Distinct sweep envelope buffers to register: {NUM_STEPS * NUM_TONES}")

# -----------------------------------------------------------------------------
# 3b. TRAP BUFFER: one composite (all 4 tones concatenated) buffer at the
#     FINAL chirp frequency, phase-continuous with the end of the sweep,
#     played once with mode='periodic' so the DAC free-runs it indefinitely
#     after the tProc halts.
# -----------------------------------------------------------------------------
trap_idata, trap_qdata, phase, trap_piece_lens = build_multitone_buffer(
    CHIRP_OFFSET_STOP_HZ, phase, tone_durations_s)
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")

total_with_trap = total_samples + len(trap_idata)
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")
assert total_with_trap <= ENV_MAXLEN, (
    f"Sweep + trap buffers exceed memory: need {total_with_trap}, "
    f"have {ENV_MAXLEN}. Reduce NUM_STEPS or CYCLE_S."
)

# -----------------------------------------------------------------------------
# 4. PROGRAM
#    For each chirp step: play tone 0 alone (periodic loop) for
#    ratio_0 * STEP_HOLD_US, then tone 1, etc. Phase continuity across the
#    switch is baked into the envelope samples themselves (phase0 threaded
#    tone-to-tone, step-to-step, and into the trap buffer), so no phase
#    register manipulation is needed at switch time.
#
#    Since tone buffers now have DIFFERENT lengths (ratio-weighted, not
#    uniform), BOTH the address register (mathi, runtime add) and the
#    packed length/mode register (regwi, compile-time constant recomputed
#    per tone from get_mode_code) must be reloaded before every tone's
#    pulse. This costs a couple of extra static instructions per tone
#    (NUM_TONES of them, independent of NUM_STEPS since it's inside the
#    hardware loop body), not per-step, so it doesn't reintroduce the
#    original tProc program-memory blowup from full unrolling.
#
#    IMPORTANT: reps must stay 1. If reps > 1, LOOP_J re-enters expecting
#    address/mode to already be at step i's tone 0, but they're only reset
#    to tone-0 values once, after LOOP_J exits (not between reps). Fine as
#    configured below; don't bump reps without restructuring the loop.
# -----------------------------------------------------------------------------
class SequentialTonePlaybackProgram(RAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i in range(cfg["num_steps"]):
            for j in range(cfg["num_tones"]):
                self.add_envelope(
                    ch=res_ch,
                    name=f"s{i}_t{j}",
                    idata=cfg["idata_pieces"][i][j],
                    qdata=cfg["qdata_pieces"][i][j],
                )

        self.add_envelope(ch=res_ch, name="trap_wfm",
                           idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])

        self.r_rp = self.ch_page(res_ch)
        self.r_addr = self.sreg(res_ch, "addr")
        self.r_mode = self.sreg(res_ch, "mode")

        spc = self.soccfg["gens"][res_ch]["samps_per_clk"]
        mgr = self._gen_mgrs[res_ch]

        # per-tone buffer length (fabric cycles) and packed mode code --
        # constant across steps for a given tone, differs between tones.
        # Both are pure Python computations from the envelope actually
        # registered via add_envelope, so they can't drift out of sync
        # with the real buffer contents.
        self.tone_wfm_len = []
        self.tone_mode_code = []
        for j in range(cfg["num_tones"]):
            pinfo = mgr.envelopes[f"s0_t{j}"]
            wfm_len = pinfo['data'].shape[0] // spc
            mc = mgr.get_mode_code(length=wfm_len, mode='periodic', outsel='input')
            self.tone_wfm_len.append(wfm_len)
            self.tone_mode_code.append(mc)
        self.addr_step_tone = self.tone_wfm_len

        # set up on tone 0 of step 0: fixes freq/phase/gain and the
        # initial addr/mode (which come out equal to tone_wfm_len[0] /
        # tone_mode_code[0] automatically, since they're derived from the
        # same envelope's stored length).
        self.set_pulse_registers(
            ch=res_ch,
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            waveform="s0_t0",
            outsel="input",
            mode="periodic",
        )

        # per-tone hold time in tProc clock cycles (same fractional split
        # every step, since TONE_RATIOS don't change across the sweep)
        self.tone_hold_cycles = [
            self.us2cycles(cfg["step_hold_us"] * frac) for frac in cfg["tone_fractions"]
        ]

        self.synci(200)

    def make_program(self):
        """One real hardware loop (expts x reps) over the chirp steps.
        Within each iteration, sequentially play all NUM_TONES tone
        buffers of the current step, each held for its own proportional
        dwell time. Before every tone after the first, step the address
        register forward by the PREVIOUS tone's buffer length and reload
        the mode register with the CURRENT tone's packed length/mode code
        (since buffer lengths now differ between tones). After the last
        tone of a step, one more address step plus a mode-reset back to
        tone 0's code lines things up for the next step."""
        p = self
        cfg = self.cfg
        res_ch = cfg["res_ch"]
        num_tones = cfg["num_tones"]
        rcount = 13
        rii = 14
        rjj = 15

        p.initialize()
        p.regwi(0, rcount, 0)
        p.regwi(0, rii, cfg['expts'] - 1)
        p.label("LOOP_I")
        p.regwi(0, rjj, cfg['reps'] - 1)
        p.label("LOOP_J")

        p.trigger(pins=[0])
        for j in range(num_tones):
            if j > 0:
                p.mathi(p.r_rp, p.r_addr, p.r_addr, '+', p.addr_step_tone[j - 1])
                p.regwi(p.r_rp, p.r_mode, p.tone_mode_code[j], f'mode tone {j}')
            p.pulse(ch=res_ch, t='auto')
            p.sync_all(p.tone_hold_cycles[j])

        p.mathi(0, rcount, rcount, "+", 1)
        p.memwi(0, rcount, p.COUNTER_ADDR)
        p.loopnz(0, rjj, 'LOOP_J')

        # advance from this step's last tone to the next step's first tone,
        # and restore tone-0's mode code (next step's first pulse expects it)
        p.mathi(p.r_rp, p.r_addr, p.r_addr, '+', p.addr_step_tone[num_tones - 1])
        p.regwi(p.r_rp, p.r_mode, p.tone_mode_code[0], 'reset mode to tone0')

        p.loopnz(0, rii, "LOOP_I")

        # --- trapping: one pulse, periodic buffer, then end(). The DAC
        # keeps cycling the 4 concatenated trap tones forever regardless
        # of what the tProc does after this, including after it halts.
        p.set_pulse_registers(
            ch=res_ch, style="arb", freq=0, phase=0, gain=cfg["gain"],
            waveform="trap_wfm", outsel="input", mode="periodic",
        )
        p.trigger(pins=[0])
        p.pulse(ch=res_ch, t='auto')
        p.end()

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "reps": 1,
    "expts": NUM_STEPS,
    "num_steps": NUM_STEPS,
    "num_tones": NUM_TONES,
    "idata_pieces": idata_pieces,
    "qdata_pieces": qdata_pieces,
    "trap_idata": trap_idata,
    "trap_qdata": trap_qdata,
    "tone_fractions": tone_fractions,
    "step_hold_us": STEP_HOLD_US,
    "gain": 32767,
}

prog = SequentialTonePlaybackProgram(soccfg, config)
prog.run(soc)  # , start_src="external"
print(f"Running on hardware — {NUM_STEPS} sweep steps, each stepping through "
      f"{NUM_TONES} tones in ratio-proportional continuous blocks "
      f"({STEP_HOLD_US:.2f} us/step, {TOTAL_SWEEP_S*1e3:.3f} ms total sweep), "
      f"then trapping (4-tone composite, periodic, indefinitely until "
      f"soc.reset_gens()).")

Generator 1: f_fabric=614.400 MHz, samps_per_clk=16, envelope sample rate=9.8304 GSPS
Envelope memory available: 65536 samples
Composite per-step time budget: 140.74 ns
  tone 0 (-76.2 MHz offset): ratio 0.337 -> buffer 47.43 ns, held 45.955 us per step
  tone 1 (-0.0 MHz offset): ratio 0.167 -> buffer 23.50 ns, held 22.773 us per step
  tone 2 (+46.7 MHz offset): ratio 0.288 -> buffer 40.53 ns, held 39.273 us per step
  tone 3 (+71.6 MHz offset): ratio 0.208 -> buffer 29.27 ns, held 28.364 us per step
Per-tone sweep buffer lengths: [480, 240, 400, 288] samples ([30.0, 15.0, 25.0, 18.0] fabric cycles)
Total envelope samples (sweep): 61952 / 65536 available
Distinct sweep envelope buffers to register: 176
Trap buffer: 1408 samples (tone slices: [480, 240, 400, 288])
Total envelope samples (sweep + trap): 63360 / 65536 available
Running on hardware — 44 sweep steps, each stepping through 4 tones in ratio-proportional continuous blocks (136.36 us/step, 6.000 ms total sweep), then trapping

In [24]:
soc.reset_gens()